In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../../")

import nest_asyncio
nest_asyncio.apply()

In [ ]:
import importlib.util
_real_find_spec = getattr(importlib.util, '_real_find_spec', importlib.util.find_spec)
importlib.util._real_find_spec = _real_find_spec
def _no_socksio(name, *a, **kw):
    if name == "socksio": return None
    return _real_find_spec(name, *a, **kw)
importlib.util.find_spec = _no_socksio

import datetime, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import psycopg2
import pytz

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder
from TB.IRSwapsTB import IRSwapsTB
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from SDRUtils._swappulse_scripts.ingest_usdswaps_tape import resolve_pg_url

warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
NY = pytz.timezone("America/New_York")


def dealer_flow_chart(
    query,
    date,
    start_hour=8,
    end_hour=17,
    n_jobs=8,
    show_tqdm=True,
):
    start = NY.localize(datetime.datetime(date.year, date.month, date.day, start_hour, 0))
    end = NY.localize(datetime.datetime(date.year, date.month, date.day, end_hour, 0))

    curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
    ts = TimeseriesBuilder()
    rate_df = ts.get_timeseries(
        start=start, end=end,
        queries=[query],
        freq="1min",
        n_jobs=n_jobs,
        routers={"IRS": IRSwapsTB(curve_mdp, show_tqdm=show_tqdm)},
    )
    if rate_df.empty:
        raise RuntimeError("No rate data returned")

    col = rate_df.columns[0]
    rate_series = rate_df[col].dropna()

    conn = psycopg2.connect(resolve_pg_url())
    directions = pd.read_sql(f"""
        SELECT execution_timestamp, dealer_direction,
               classification_method, direction_confidence,
               structure_dv01, rate_index_clean, trade_type
        FROM arbs_stir_direction_v1
        WHERE execution_timestamp::date = '{date.isoformat()}'
          AND dealer_direction IN ('PAID', 'RECEIVED')
        ORDER BY execution_timestamp
    """, conn)
    conn.close()

    directions["execution_timestamp"] = pd.to_datetime(directions["execution_timestamp"])
    if rate_series.index.tz is not None:
        if directions["execution_timestamp"].dt.tz is None:
            directions["execution_timestamp"] = directions["execution_timestamp"].dt.tz_localize("UTC")
        directions["execution_timestamp"] = directions["execution_timestamp"].dt.tz_convert(rate_series.index.tz)
    mask = (directions["execution_timestamp"] >= rate_series.index.min()) & \
           (directions["execution_timestamp"] <= rate_series.index.max())
    directions = directions[mask]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=rate_series.index, y=rate_series.values,
        mode="lines", line=dict(width=2, color="#898781"),
        name=str(col), hovertemplate="%{y:.4f}<extra>rate</extra>",
    ))

    for direction, symbol, color in [
        ("RECEIVED", "triangle-up", "#008300"),
        ("PAID", "triangle-down", "#e34948"),
    ]:
        sub = directions[directions["dealer_direction"] == direction]
        if sub.empty:
            continue
        y_vals = []
        for t in sub["execution_timestamp"]:
            idx = rate_series.index.get_indexer([t], method="nearest")
            y_vals.append(float(rate_series.iloc[idx[0]]) if idx[0] != -1 else np.nan)

        sizes = np.clip(sub["structure_dv01"].fillna(0).values / 5000, 4, 25)

        fig.add_trace(go.Scatter(
            x=sub["execution_timestamp"], y=y_vals,
            mode="markers",
            marker=dict(symbol=symbol, size=sizes, color=color,
                        line=dict(width=1, color="white")),
            name=direction,
            customdata=np.column_stack([
                sub["structure_dv01"].fillna(0).values,
                sub["direction_confidence"].fillna("?").values,
                sub["classification_method"].fillna("?").values,
                sub["trade_type"].fillna("?").values,
            ]),
            hovertemplate=(
                f"{direction}<br>"
                "DV01: %{customdata[0]:,.0f}<br>"
                "conf: %{customdata[1]}<br>"
                "method: %{customdata[2]}<br>"
                "type: %{customdata[3]}<extra></extra>"
            ),
        ))

    tenor_label = getattr(query, 'tenor', str(col))
    fig.update_layout(
        title=f"{tenor_label} — {date.isoformat()} dealer flow",
        yaxis_title="rate (bps)" if rate_series.max() < 1 else "rate (%)",
        xaxis_title="",
        height=500,
        plot_bgcolor="#fcfcfb",
        paper_bgcolor="white",
        font=dict(family="system-ui, -apple-system, sans-serif", color="#0b0b0b"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hovermode="x unified",
        yaxis=dict(gridcolor="#e1e0d9", gridwidth=1),
        xaxis=dict(gridcolor="#e1e0d9", gridwidth=1),
    )
    return fig

In [4]:
q = UnifiedQuery(
    curve="USD-OIS-Q12xM12STIRT-SERFFX-MIX23",
    tenor="fomc_jul26",
    value=UnifiedValue.IRS_RATE,
)

fig = dealer_flow_chart(q, datetime.date(2026, 7, 2))
fig.show()

RecursionError: maximum recursion depth exceeded